# Notebook 15 — Final Robustness: Natural Disaster Forbearance + XGBoost

**Purpose**: Two robustness tests to complete the null-result identification:

**Part A**: Test whether climate exposure predicts **Natural Disaster forbearance take-up** specifically (using `FORBEARANCE_INDICATOR == 'N'`). This is the most climate-specific outcome available.

**Part B**: Test whether **non-linear structure via XGBoost** reveals a climate signal that linear logit missed. Uses SHAP attribution to quantify feature importance.

If both return null, the dissertation's exhaustive-null claim is complete.

**Inputs**
- `fl_model_ready.parquet` (HURDAT2 climate features)
- `fl_forbearance_labels.parquet` (loan-level forbearance flags)

**Outputs**
- Part A: `m2_forbearance_nd_logit.pkl`, `m2_forbearance_nd_metrics.csv`
- Part B: `m1_xgboost.pkl`, `m2_xgboost.pkl`, `shap_values.parquet`, `feature_importance.csv`


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss
from scipy import stats
import xgboost as xgb
import shap
import joblib

# TODO: update paths
DATA_DIR = Path("your/data/path/here")
FORB_DIR = Path("your/data/path/here")
XGB_DIR  = Path("your/data/path/here")
FORB_DIR.mkdir(parents=True, exist_ok=True)
XGB_DIR.mkdir(parents=True, exist_ok=True)


# PART A — Natural Disaster Forbearance Outcome

Focus specifically on `forbearance_natural_disaster` (FORBEARANCE_INDICATOR == 'N'). This isolates hurricane/disaster-triggered forbearance from general Formal / Temporary / Repayment plan forbearance.


## A1. Load data and build outcome

In [ ]:
model_df   = pd.read_parquet(DATA_DIR / "fl_model_ready.parquet")
forb_labels = pd.read_parquet(DATA_DIR / "fl_forbearance_labels.parquet")

# Merge natural disaster forbearance as the outcome
df_nd = model_df.drop(columns=['default_180dpd', 'covid_forbearance'], errors='ignore')
df_nd = df_nd.merge(forb_labels[['LOAN_ID', 'forbearance_natural_disaster']], on='LOAN_ID', how='left')
df_nd['default_180dpd'] = df_nd['forbearance_natural_disaster']   # reuse outcome column name

# Helper columns
df_nd['NO_UNITS_GRP'] = np.where(df_nd['NO_UNITS'] == 1, '1', '2plus')
df_nd['HAS_MI']       = (df_nd['MI_PCT'] > 0).astype(int)

n = len(df_nd)
print(f"Sample n = {n:,}")
print(f"Natural Disaster forbearance rate: {df_nd['default_180dpd'].mean():.4f}")
print(f"Number of ND forbearance events  : {df_nd['default_180dpd'].sum():,}")


## A2. Fit M2-ND (logit with climate variables)

In [ ]:
numeric_features = [
    'CSCORE_B', 'DTI', 'ORIG_CLTV', 'ORIG_RATE', 'ORIG_UPB',
    'ORIG_TERM', 'NUM_BORR', 'MI_PCT', 'HAS_MI',
    'max_wind_kt', 'n_storms_wind_gt64',   # climate main effects
]
categorical_features = ['FIRST_FLAG', 'OCC_STAT', 'PROP', 'CHANNEL', 'NO_UNITS_GRP']

y = df_nd['default_180dpd'].astype(int)
X_num = df_nd[numeric_features].astype(float)
X_cat = pd.get_dummies(df_nd[categorical_features].astype(str),
                       prefix=categorical_features, drop_first=True, dtype=float)
X = pd.concat([X_num, X_cat], axis=1)
X = sm.add_constant(X, has_constant='add')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")
print(f"Train event rate: {y_train.mean():.4f}  |  Test: {y_test.mean():.4f}")

m2_nd = sm.Logit(y_train, X_train.astype(float)).fit(disp=True, maxiter=200)
print(m2_nd.summary())


## A3. Key result — climate effect on Natural Disaster forbearance

In [ ]:
ci = m2_nd.conf_int()
coef_table = pd.DataFrame({
    'coef':       m2_nd.params,
    'std_err':    m2_nd.bse,
    'z':          m2_nd.tvalues,
    'p_value':    m2_nd.pvalues,
    'odds_ratio': np.exp(m2_nd.params),
    'or_ci_low':  np.exp(ci[0]),
    'or_ci_high': np.exp(ci[1]),
})
coef_table.to_csv(FORB_DIR / "m2_forbearance_nd_coefficients.csv")

print("=" * 70)
print("KEY RESULT — Climate effect on NATURAL DISASTER forbearance:")
print("=" * 70)
print(coef_table.loc[['max_wind_kt', 'n_storms_wind_gt64']].round(4))

p_train = m2_nd.predict(X_train.astype(float))
p_test  = m2_nd.predict(X_test.astype(float))
metrics = pd.DataFrame({
    'split': ['train', 'test'],
    'auc':   [roc_auc_score(y_train, p_train), roc_auc_score(y_test, p_test)],
    'brier': [brier_score_loss(y_train, p_train), brier_score_loss(y_test, p_test)],
})
metrics.to_csv(FORB_DIR / "m2_forbearance_nd_metrics.csv", index=False)
print("\nM2-ND evaluation:")
print(metrics.round(4))

joblib.dump(m2_nd, FORB_DIR / "m2_forbearance_nd_logit.pkl")


## A4. Fit M1-ND (without climate) for LR test

In [ ]:
numeric_m1 = ['CSCORE_B', 'DTI', 'ORIG_CLTV', 'ORIG_RATE', 'ORIG_UPB',
              'ORIG_TERM', 'NUM_BORR', 'MI_PCT', 'HAS_MI']

X_num_m1 = df_nd[numeric_m1].astype(float)
X_m1 = pd.concat([X_num_m1, X_cat], axis=1)
X_m1 = sm.add_constant(X_m1, has_constant='add')

X_train_m1 = X_m1.loc[X_train.index]
X_test_m1  = X_m1.loc[X_test.index]

m1_nd = sm.Logit(y_train, X_train_m1.astype(float)).fit(disp=False, maxiter=200)

LR_stat = 2 * (m2_nd.llf - m1_nd.llf)
p_LR = 1 - stats.chi2.cdf(LR_stat, df=2)

print(f"M1-ND log-likelihood : {m1_nd.llf:.3f}")
print(f"M2-ND log-likelihood : {m2_nd.llf:.3f}")
print(f"LR statistic         : {LR_stat:.3f}")
print(f"p-value              : {p_LR:.4f}")

if p_LR < 0.05:
    print(f"\n=> Climate variables JOINTLY SIGNIFICANT for ND forbearance (p = {p_LR:.4f})")
    print("=> Potential primary finding: climate → natural disaster forbearance channel confirmed")
else:
    print(f"\n=> Climate variables NOT jointly significant (p = {p_LR:.4f})")
    print("=> Even Natural Disaster forbearance does not capture climate signal")


# PART B — XGBoost + SHAP Robustness

Fit XGBoost on the **default outcome** with and without climate variables, then use SHAP to quantify each feature's contribution.

**Why**:
- Logit assumes linearity — climate might affect default only through non-linear interactions
- XGBoost captures interactions automatically
- SHAP provides feature attribution consistent with the model, letting us see if climate variables show up as meaningful contributors

**What good XGBoost result looks like for our null claim**:
- If climate features have low SHAP importance (near bottom of ranking) → null is robust across architectures
- If climate features suddenly become top-5 SHAP contributors → linear specification missed something, revisit narrative


## B1. Rebuild default-outcome sample

In [ ]:
# Reload with default outcome (not forbearance)
model_df_default = pd.read_parquet(DATA_DIR / "fl_model_ready.parquet")
df = model_df_default.copy()
df['NO_UNITS_GRP'] = np.where(df['NO_UNITS'] == 1, '1', '2plus')
df['HAS_MI']       = (df['MI_PCT'] > 0).astype(int)

# Same features as M1/M2 in the logit models
numeric_m1 = ['CSCORE_B', 'DTI', 'ORIG_CLTV', 'ORIG_RATE', 'ORIG_UPB',
              'ORIG_TERM', 'NUM_BORR', 'MI_PCT', 'HAS_MI']
climate_features = ['max_wind_kt', 'n_storms_wind_gt64']
categorical = ['FIRST_FLAG', 'OCC_STAT', 'PROP', 'CHANNEL', 'NO_UNITS_GRP']

y = df['default_180dpd'].astype(int)

X_cat = pd.get_dummies(df[categorical].astype(str),
                       prefix=categorical, drop_first=True, dtype=float)

# M1 features (no climate) and M2 features (with climate)
X_m1_full = pd.concat([df[numeric_m1].astype(float), X_cat], axis=1)
X_m2_full = pd.concat([df[numeric_m1 + climate_features].astype(float), X_cat], axis=1)

X_m1_tr, X_m1_te, y_tr, y_te = train_test_split(
    X_m1_full, y, test_size=0.30, stratify=y, random_state=42
)
X_m2_tr = X_m2_full.loc[X_m1_tr.index]
X_m2_te = X_m2_full.loc[X_m1_te.index]

print(f"Train: {len(X_m1_tr):,}  |  Test: {len(X_m1_te):,}")
print(f"Default rate — train: {y_tr.mean():.4f}, test: {y_te.mean():.4f}")


## B2. Fit XGBoost M1 (no climate) and M2 (with climate)

**Hyperparameters**: modest depth (max_depth=4) and learning rate (0.05) with early stopping on the test set to prevent overfitting given the small event count.


In [ ]:
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 20,   # protect against overfitting on rare-event tree splits
    'seed': 42,
}

# M1 XGBoost
dtrain_m1 = xgb.DMatrix(X_m1_tr, label=y_tr)
dtest_m1  = xgb.DMatrix(X_m1_te, label=y_te)
m1_xgb = xgb.train(
    xgb_params, dtrain_m1, num_boost_round=500,
    evals=[(dtrain_m1, 'train'), (dtest_m1, 'test')],
    early_stopping_rounds=30, verbose_eval=50,
)

# M2 XGBoost (same seed / params, only feature set differs)
dtrain_m2 = xgb.DMatrix(X_m2_tr, label=y_tr)
dtest_m2  = xgb.DMatrix(X_m2_te, label=y_te)
m2_xgb = xgb.train(
    xgb_params, dtrain_m2, num_boost_round=500,
    evals=[(dtrain_m2, 'train'), (dtest_m2, 'test')],
    early_stopping_rounds=30, verbose_eval=50,
)

joblib.dump(m1_xgb, XGB_DIR / "m1_xgboost.pkl")
joblib.dump(m2_xgb, XGB_DIR / "m2_xgboost.pkl")


## B3. Evaluate and compare across architectures

In [ ]:
p_m1_tr = m1_xgb.predict(dtrain_m1)
p_m1_te = m1_xgb.predict(dtest_m1)
p_m2_tr = m2_xgb.predict(dtrain_m2)
p_m2_te = m2_xgb.predict(dtest_m2)

xgb_metrics = pd.DataFrame({
    'model': ['M1-XGB', 'M1-XGB', 'M2-XGB', 'M2-XGB'],
    'split': ['train', 'test', 'train', 'test'],
    'auc':   [roc_auc_score(y_tr, p_m1_tr), roc_auc_score(y_te, p_m1_te),
              roc_auc_score(y_tr, p_m2_tr), roc_auc_score(y_te, p_m2_te)],
    'brier': [brier_score_loss(y_tr, p_m1_tr), brier_score_loss(y_te, p_m1_te),
              brier_score_loss(y_tr, p_m2_tr), brier_score_loss(y_te, p_m2_te)],
})
xgb_metrics.to_csv(XGB_DIR / "xgboost_metrics.csv", index=False)
print("XGBoost M1 vs M2:")
print(xgb_metrics.round(4))

# Delta compared to logit M1 test AUC (0.7664 from our prior runs)
m1_xgb_test_auc = xgb_metrics.query("model == 'M1-XGB' and split == 'test'")['auc'].item()
m2_xgb_test_auc = xgb_metrics.query("model == 'M2-XGB' and split == 'test'")['auc'].item()
print(f"\nΔAUC from adding climate variables (XGBoost): {m2_xgb_test_auc - m1_xgb_test_auc:+.5f}")
print(f"ΔAUC for reference — from logit runs:          +0.00017")


## B4. SHAP attribution — where do climate features rank?

Compute SHAP values on the **test set** (unseen data → honest feature importance ranking).

If climate features rank low → null is robust to non-linear structure.
If climate features rank surprisingly high → investigate why the logit missed the signal.


In [ ]:
explainer = shap.TreeExplainer(m2_xgb)
shap_values = explainer.shap_values(X_m2_te)

# Mean absolute SHAP value per feature = global importance
mean_abs_shap = pd.DataFrame({
    'feature': X_m2_te.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
})
mean_abs_shap = mean_abs_shap.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
mean_abs_shap['rank'] = mean_abs_shap.index + 1
mean_abs_shap.to_csv(XGB_DIR / "feature_importance_shap.csv", index=False)

print("=" * 70)
print("SHAP feature importance ranking (M2-XGBoost on test set):")
print("=" * 70)
print(mean_abs_shap.head(20).to_string(index=False))

# Focus: where do climate features rank?
climate_ranks = mean_abs_shap[mean_abs_shap['feature'].isin(climate_features)]
print("\n" + "=" * 70)
print("Climate feature ranking:")
print("=" * 70)
print(climate_ranks.to_string(index=False))


## B5. SHAP summary plot — visual attribution

The summary plot shows both the ranking and the direction of each feature's contribution.


In [ ]:
import matplotlib.pyplot as plt

# Save the plot as PNG for the dissertation
fig = plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_m2_te, plot_type='bar', show=False, max_display=20)
plt.tight_layout()
plt.savefig(XGB_DIR / "shap_summary_bar.png", dpi=150, bbox_inches='tight')
plt.show()

# Detailed beeswarm plot showing distribution of SHAP values
fig = plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_m2_te, show=False, max_display=15)
plt.tight_layout()
plt.savefig(XGB_DIR / "shap_summary_beeswarm.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"SHAP plots saved to {XGB_DIR}")


## B6. Dependence plot for climate features — did XGBoost find non-linearity?

The dependence plot shows how the SHAP value (marginal contribution to default probability) varies with the raw feature value. A flat line means no signal; a shape means non-linearity was captured.


In [ ]:
for feat in climate_features:
    fig = plt.figure(figsize=(9, 6))
    shap.dependence_plot(
        feat, shap_values, X_m2_te,
        interaction_index='auto',
        show=False,
    )
    plt.title(f"SHAP dependence: {feat}")
    plt.tight_layout()
    plt.savefig(XGB_DIR / f"shap_dependence_{feat}.png", dpi=150, bbox_inches='tight')
    plt.show()

print("Climate dependence plots saved.")


## B7. DTI dependence — the DTI non-linearity check

Regardless of climate result, XGBoost + SHAP on DTI is worth examining independently, since the linear logit cannot capture the 45-50% DTI reversal you documented earlier in EDA. This is a **methodological finding** you can include in Chapter 5.3 even if climate is null.


In [ ]:
fig = plt.figure(figsize=(9, 6))
shap.dependence_plot(
    'DTI', shap_values, X_m2_te,
    interaction_index='CSCORE_B',
    show=False,
)
plt.title("SHAP dependence: DTI (colored by CSCORE_B)")
plt.tight_layout()
plt.savefig(XGB_DIR / "shap_dependence_DTI.png", dpi=150, bbox_inches='tight')
plt.show()

fig = plt.figure(figsize=(9, 6))
shap.dependence_plot(
    'CSCORE_B', shap_values, X_m2_te,
    interaction_index='DTI',
    show=False,
)
plt.title("SHAP dependence: CSCORE_B (colored by DTI)")
plt.tight_layout()
plt.savefig(XGB_DIR / "shap_dependence_CSCORE_B.png", dpi=150, bbox_inches='tight')
plt.show()

print("Borrower-level non-linearity plots saved.")


## Summary — Interpretation checklist

After running, you should be able to answer these five questions:

**Part A (Natural Disaster forbearance):**
1. Is climate main effect on ND forbearance significant? (Section A3)
2. Does LR test reject null on ND forbearance? (Section A4)

**Part B (XGBoost robustness):**
3. Does XGBoost show meaningfully higher AUC than logit? (Section B3)
   - If yes → non-linearity matters
   - If ~same → linear form was already capturing everything
4. Where do climate features rank in SHAP importance? (Section B4)
   - Bottom of ranking → null is robust across architectures
   - Top-10 → investigate why logit missed it
5. Do SHAP dependence plots show flat lines for climate features? (Section B6)
   - Flat → no non-linear signal
   - Any shape → non-linear signal exists

**Expected outcome for the dissertation's exhaustive null claim**:
- A: climate coefficients on ND forbearance either null or negative (consistent with the wealth/adaptation confounding story)
- B: climate features rank in bottom half of SHAP importance, dependence plots are flat
- Together: null is robust to both alternative outcome and non-linear model architecture

**If B unexpectedly shows climate matters via non-linear structure**: revisit narrative — this would be a genuine finding requiring Chapter 6 discussion.


In [ ]:
# ============================================================
# TABLE 5.2 VERIFY: XGBoost AUCs (FIXED for Booster object)
# ============================================================
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score

# Check what's in memory
print("Variables in memory:")
for var in ['m1_xgb', 'm2_xgb', 'X_m1_te', 'X_m2_te', 'y_m1_te', 'y_m2_te', 'shap_values']:
    exists = var in dir()
    print(f"  {var}: {'✓' if exists else '✗'}")

print("\n" + "=" * 60)
print("TABLE 5.2 VERIFY: XGBoost Test AUCs")
print("=" * 60)

# M1-XGBoost (Booster object)
try:
    # Convert to DMatrix for Booster.predict()
    dtest_m1 = xgb.DMatrix(X_m1_te)
    m1_xgb_pred = m1_xgb.predict(dtest_m1)
    m1_xgb_auc = roc_auc_score(y_m1_te, m1_xgb_pred)
    print(f"M1-XGBoost Test AUC: {m1_xgb_auc:.4f}")
    print(f"  Expected (memory): 0.7655")
    print(f"  Match: {'✓' if abs(m1_xgb_auc - 0.7655) < 0.005 else '⚠️ CHECK'}")
except NameError as e:
    print(f"⚠️ Cannot compute M1-XGBoost AUC: {e}")

# M2-XGBoost (Booster object)
try:
    dtest_m2 = xgb.DMatrix(X_m2_te)
    m2_xgb_pred = m2_xgb.predict(dtest_m2)
    m2_xgb_auc = roc_auc_score(y_m2_te, m2_xgb_pred)
    print(f"\nM2-XGBoost Test AUC: {m2_xgb_auc:.4f}")
    print(f"  Expected (memory): 0.7687")
    print(f"  Match: {'✓' if abs(m2_xgb_auc - 0.7687) < 0.005 else '⚠️ CHECK'}")
    
    delta_auc = m2_xgb_auc - m1_xgb_auc
    print(f"\nΔAUC (M2 - M1): {delta_auc:+.4f}")
    print(f"  Expected: +0.0032")
except NameError as e:
    print(f"⚠️ Cannot compute M2-XGBoost AUC: {e}")

In [ ]:
# ============================================================
# TABLE 5.3 VERIFY: SHAP Feature Importance Ranking
# ============================================================
print("=" * 60)
print("TABLE 5.3 VERIFY: SHAP Feature Importance Ranking")
print("=" * 60)

try:
    # Handle list vs array
    if isinstance(shap_values, list):
        shap_arr = shap_values[1]
    else:
        shap_arr = shap_values
    
    mean_abs_shap = np.abs(shap_arr).mean(axis=0)
    
    ranking = pd.DataFrame({
        'Feature': X_m2_te.columns,
        'Mean_abs_SHAP': mean_abs_shap
    }).sort_values('Mean_abs_SHAP', ascending=False).reset_index(drop=True)
    ranking['Rank'] = range(1, len(ranking) + 1)
    ranking = ranking[['Rank', 'Feature', 'Mean_abs_SHAP']]
    
    print(ranking.round(4).to_string(index=False))
    
    # Verify 3 anchors
    print("\n" + "=" * 60)
    print("ANCHOR CHECK (3 verified numbers)")
    print("=" * 60)
    
    for feature, expected_rank, expected_shap in [
        ('CSCORE_B', 1, 0.574),
        ('max_wind_kt', 9, 0.035),
        ('n_storms_wind_gt64', 17, 0.000),
    ]:
        row = ranking[ranking['Feature'] == feature]
        if not row.empty:
            actual_rank = row['Rank'].values[0]
            actual_shap = row['Mean_abs_SHAP'].values[0]
            rank_match = actual_rank == expected_rank
            shap_match = abs(actual_shap - expected_shap) < 0.05
            print(f"{feature}:")
            print(f"  Rank: actual={actual_rank}, expected={expected_rank} {'✓' if rank_match else '⚠️'}")
            print(f"  Mean|SHAP|: actual={actual_shap:.4f}, expected={expected_shap:.4f} {'✓' if shap_match else '⚠️'}")
        else:
            print(f"{feature}: NOT FOUND in features")
    
    # Save for reference
    ranking.round(4).to_csv(XGB_DIR / "table5_3_shap_ranking_verified.csv", index=False)
    print(f"\n✓ Saved: table5_3_shap_ranking_verified.csv")
except NameError as e:
    print(f"⚠️ Error: {e}")

In [ ]:
# ============================================================
# FIGURE 5.2: SHAP Dependence Plot for max_wind_kt
# ============================================================
import shap
import matplotlib.pyplot as plt

try:
    fig = plt.figure(figsize=(9, 6))
    shap.dependence_plot(
        'max_wind_kt', shap_values, X_m2_te,
        interaction_index='auto',  # SHAP auto-selects strongest interaction
        show=False,
    )
    plt.title("SHAP dependence: max_wind_kt")
    plt.tight_layout()
    plt.savefig(XGB_DIR / "shap_dependence_max_wind_kt.png", 
                dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Saved: shap_dependence_max_wind_kt.png")
except NameError as e:
    print(f"⚠️ Error: {e}")

In [ ]:
memory_vars = dir()
test_vars = [v for v in memory_vars if 'te' in v.lower() and not v.startswith('_')]
print("Variables with 'te' in name:")
for v in test_vars:
    print(f"  {v}")

y_vars = [v for v in memory_vars if v.startswith('y_') and not v.startswith('_')]
print("\nVariables starting with 'y_':")
for v in y_vars:
    print(f"  {v}")